# 15. More seeds of the target-encoded model

Seed 42 is ledger row 17. This runs seeds 2024, 7, 2025 and 13 with everything
else identical, so the five can be blended the way experiments 9 to 15 were.

On the raw feature set a 5-seed rank blend was worth +0.000546 over a single seed,
and the step halved with each added seed. The members here are ~0.0033 stronger and
their fold spread is tighter, so that number is re-measured rather than assumed.

The encoder, the folds and the leak checks are unchanged from `13`. One kernel
rather than four pushes: four seeds at about nine minutes each.


In [ ]:
# Set False for the real run. Smoke mode exercises every line on 20k rows.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# Ledger row 9: the same config with the raw feature set. This is the number the
# experiment is measured against, and the comparison is paired per fold.
BASELINE_NAME = "lgbm_bag08_seed42"
BASELINE_CV = 0.963471
EXPECTED_FOLD_SHA = "ec282b0968059676"

PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.05, n_estimators=2000,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1,
    deterministic=True, force_row_wise=True,
)
print(f"SMOKE = {SMOKE}")


In [ ]:
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold


def locate(name):
    kag = Path("/kaggle/input")
    if kag.exists():
        hits = sorted(kag.rglob(name))
        if hits:
            return hits[0]
    for b in [Path.cwd(), *Path.cwd().parents]:
        p = b / "data" / "raw" / name
        if p.exists():
            return p
    raise FileNotFoundError(name)


train = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train.columns if c not in ("id", TARGET)]

if SMOKE:
    train = train.sample(20000, random_state=0).reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    PARAMS["n_estimators"] = 200

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print(f"rows {len(train)}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")


In [ ]:
# Seed 42 already exists as ledger row 17, so it is not re-run here.
SEEDS = [2024, 7, 2025, 13]
print("seeds to run:", SEEDS)


In [ ]:
results = {}
t0 = time.time()

for s in SEEDS:
    P = dict(PARAMS)
    P["random_state"] = s
    oof = np.zeros(len(train))
    tst = np.zeros(len(test))
    sc = []
    for f in range(5):
        tr = np.where(folds != f)[0]
        va = np.where(folds == f)[0]
        # The encoder's own inner split is deliberately NOT reseeded. Only the
        # model's stochasticity varies, so this is a seed sweep of the model and
        # not of the representation as well.
        Xtr, Xva, Xte = build(X, y, tr, va, X_test)
        m = lgb.LGBMClassifier(**P).fit(Xtr, y[tr])
        oof[va] = m.predict_proba(Xva)[:, 1]
        tst += m.predict_proba(Xte)[:, 1] / 5
        sc.append(roc_auc_score(y[va], oof[va]))
    results[s] = (float(np.mean(sc)), float(np.std(sc)), oof, tst)
    print(f"seed {s:5d}: CV {np.mean(sc):.6f} +/- {np.std(sc):.6f}"
          f"   [{(time.time() - t0) / 60:.1f} min total]")


In [ ]:
prefix = "SMOKE_" if SMOKE else ""
out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
for s, (cv, sd, oof, tst) in results.items():
    np.save(out / f"{prefix}te_seed{s}_oof.npy", oof)
    np.save(out / f"{prefix}te_seed{s}_test.npy", tst)

print("saved", len(results), "seed vectors")
print()
print("Blending against seed 42 is done locally, where its vector lives.")
for s, (cv, sd, _, _) in results.items():
    print(f"  ledger line: lgbm_bag08_seed{s}_te   cv {cv:.6f} +/- {sd:.6f}")
